# Lesson 75 · 🎨 The ASR pipeline drawn — seeing the road from speech to text: waveform → spectrogram → token alignment, visualized end to end

**Goal**: draw the whole road from speech to text, so the waveform, spectrogram, token alignment and attention can all be read at a glance.

> **🎨**: look at the alignment plot first, then the heatmap; the point of this lesson is learning to *read* the pictures.

🔗 **Aurora connection**:
- `aurora.audio.mel` — turns the raw waveform into a mel spectrogram and feeds it to the Whisper encoder
- The Whisper encoder outputs a hidden-state vector for every frame
- Each output token of the Whisper decoder takes a weighted sum of those hidden states via cross-attention; this lesson visualizes that weight matrix

← **Previous lesson** · [L74 · ASR error analysis](L74_asr_error_analysis.ipynb)

> Last lesson covered **ASR error analysis**: substitution / deletion / insertion patterns, from a WER number to what to fix.
> This lesson explores **the ASR pipeline drawn**.

## Learning goals

By the end of this lesson you should be able to:

1. **Explain the CTC decoding rule**: trace a given path by hand (collapse consecutive repeats first, then remove blanks) and decide whether it decodes to the target sequence.
2. **Read a cross-attention heatmap**: use how clean the diagonal is, and the attention entropy, to judge the Whisper decoder's alignment quality and its risk of hallucination.
3. **Quantify alignment quality**: compute the mean entropy of the attention-weight matrix, and understand that higher entropy means the decoder is less sure which audio frames the current token belongs to.

## Key intuition for this lesson

An ASR system outputs a discrete sequence of tokens, but its input is a continuous stream of audio frames — the two sequences have completely different lengths, and aligning them is the core problem.
CTC adds a blank symbol and then sums the probabilities of all valid paths, so the model can be trained without anyone aligning the data by hand.
Attention, on the other hand, decides for itself during decoding "which audio frames should the current output token look at most", and a heatmap makes this invisible alignment plain to see.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product


## 0. The starting point: waveform → spectrogram

The road in the title runs "waveform → spectrogram → tokens → text", so let's honestly start at the beginning. First we synthesize a tiny fake "HI" with `aurora.audio`: a chirp sliding upward in frequency stands in for the transition of the consonant H, followed by a steady tone with a few harmonics standing in for the vowel I, with a bit of silence on either side. Then the `mel_spectrogram` you wrote yourself in L46–L47 turns it into a log-mel spectrogram.

Two things to watch in the picture:

- **How much the time axis gets squeezed**: 16000 samples per second become just a few dozen frames in the spectrogram (hop=160 samples, 10 ms per frame). Frames like these, not raw samples, are what the Whisper encoder eats.
- **Frame count ≠ token count**: the spectrogram has dozens of frames, but the target "HI" has only 2 tokens. Which goes with which? That's exactly what the CTC lattice in the next section settles, and its horizontal axis is these frames (shrunk to T=6 below so it fits on the page).


In [ ]:
# Step 0: draw the start of the road — waveform → log-mel spectrogram
# Synthesize a fake "HI"-like utterance with aurora.audio: an upward chirp as the consonant H's transition, a steady tone with harmonics as the vowel I
from aurora.audio import sine, chirp, mel_spectrogram, power_to_db

SR = 16000
HOP = 160                      # one frame every 10 ms (the Whisper front end uses this hop too)
silence = np.zeros(int(0.15 * SR))
h_part = chirp(300, 900, duration=0.20, sample_rate=SR, amplitude=0.5)          # "H": a fast upward frequency glide
i_part = sum(sine(f, duration=0.30, sample_rate=SR, amplitude=a)                # "I": fundamental 250 Hz + a few harmonics
             for f, a in [(250, 0.5), (500, 0.3), (750, 0.2), (2500, 0.15)])
fade = np.linspace(0, 1, int(0.01 * SR))                                         # 10 ms fade in/out so the seams don't click
for part in (h_part, i_part):
    part[:len(fade)] *= fade
    part[-len(fade):] *= fade[::-1]
wave = np.concatenate([silence, h_part, i_part, silence])
t_axis = np.arange(len(wave)) / SR

mel = mel_spectrogram(wave, sample_rate=SR, n_fft=512, hop_length=HOP, n_mels=40)   # (n_frames, 40)
mel_db = power_to_db(mel, ref=mel.max())

fig, (ax0, ax1) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax0.plot(t_axis, wave, lw=0.6)
ax0.set_ylabel('amplitude')
ax0.set_title(f'synthetic speech "HI": {len(wave)} samples @ {SR} Hz = {len(wave)/SR:.2f} s')
ax1.imshow(mel_db.T, origin='lower', aspect='auto', cmap='magma',
           extent=[0, len(wave) / SR, 0, mel.shape[1]])
ax1.set_ylabel('mel band')
ax1.set_xlabel('time (s)')
ax1.set_title(f'log-mel spectrogram: {mel.shape[0]} frames × {mel.shape[1]} mel bands (hop={HOP} samples = {HOP/SR*1000:.0f} ms)')
for x_mark, name in [(0.15, 'H'), (0.35, 'I'), (0.65, '')]:
    ax0.axvline(x_mark, color='gray', ls='--', lw=0.8)
    ax1.axvline(x_mark, color='white', ls='--', lw=0.8)
    if name:
        ax1.text(x_mark + 0.01, 36, name, color='white', fontsize=12, weight='bold')
plt.tight_layout()
plt.show()

print(f'waveform: {len(wave)} samples → spectrogram: {mel.shape[0]} frames (the time axis shrank {len(wave) / mel.shape[0]:.0f}×)')
print(f'these {mel.shape[0]} frames are the horizontal axis of the CTC lattice below; the target "HI" has only 2 tokens: {mel.shape[0]} versus 2 is the alignment problem')


## 1. CTC alignment: many paths → the same target sequence

CTC (Connectionist Temporal Classification) outputs one character at every time step (including a special blank symbol `-`).
Decoding rule: first collapse consecutive repeats, then remove every blank, and what's left is the final sequence.

For the target word "HI", all of the following paths are valid:

```
Example paths (T=6, vocab: H I -; the code below also uses T=6 and shows more paths)
  H H - I - -   -> collapse -> H - I -     -> remove blanks -> HI  ✓
  H - I - - -   -> collapse -> H - I -     -> remove blanks -> HI  ✓
  - H - I - -   -> collapse -> - H - I -   -> remove blanks -> HI  ✓
  H - - I I -   -> collapse -> H - I -     -> remove blanks -> HI  ✓
```
> ⚠️ Careful: the path `H H I - I` decodes to "HII" (two separate I's) — not valid for "HI"!
> A blank between two I's is exactly what makes them count as two separate tokens; without the blank they would collapse into one.

CTC loss = `-log( sum of the probabilities of all valid paths )`, and the forward algorithm sums this efficiently over the lattice.
The figure below colors every node that valid paths pass through, to help you feel how explosively the "number of valid paths" grows with T.


In [ ]:
# Visualizing valid CTC paths (a static lattice snapshot)
# Target sequence: 'HI', time steps T=6, vocab: {H:0, I:1, -:2(blank)}
# Enumerate every path of length T and pick out the valid ones that decode to 'HI'

VOCAB = ['H', 'I', '-']
TARGET = ['H', 'I']
T = 6
V = len(VOCAB)


def ctc_decode(path):
    # CTC decode: collapse consecutive repeats -> remove blanks
    collapsed = [path[0]] + [path[i] for i in range(1, len(path)) if path[i] != path[i-1]]
    return [c for c in collapsed if c != '-']


all_paths = list(product(range(V), repeat=T))
valid_paths = [p for p in all_paths if ctc_decode([VOCAB[c] for c in p]) == TARGET]
print(f'Total paths: {V}^{T} = {V**T}')
print(f'Valid paths (target="HI"): {len(valid_paths)}')
print(f'First 5 valid paths: {[tuple(VOCAB[c] for c in p) for p in valid_paths[:5]]}')

# Draw the lattice: nodes (t, v); nodes that valid paths pass through are shown in red
node_visited = np.zeros((T, V), dtype=int)
for p in valid_paths:
    for t, v in enumerate(p):
        node_visited[t, v] += 1

fig, ax = plt.subplots(figsize=(9, 3))
for t in range(T):
    for v in range(V):
        count = node_visited[t, v]
        color = plt.cm.Reds(min(count / len(valid_paths) * 3, 1.0)) if count > 0 else '#e0e0e0'
        ax.add_patch(plt.Circle((t, v), 0.35, color=color, zorder=2))
        ax.text(t, v, VOCAB[v], ha='center', va='center', fontsize=10, zorder=3,
                color='white' if count > 0 else '#aaa')

ax.set_xlim(-0.6, T - 0.4)
ax.set_ylim(-0.6, V - 0.4)
ax.set_xticks(range(T))
ax.set_xticklabels([f't={i}' for i in range(T)])
ax.set_yticks(range(V))
ax.set_yticklabels(VOCAB)
ax.set_title(f'CTC lattice (target="HI", T={T})\nred intensity = how often valid paths pass through the node')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()
print(f'Share of valid paths: {len(valid_paths)/V**T*100:.1f}%')

### Exercise 1: write `ctc_decode` by hand and check it

Without looking at the implementation above, write your own `my_ctc_decode(path)` function so that it passes the tests below.

**CTC rules, recap**:
1. First collapse consecutive repeated characters (blank or not, it doesn't matter)
2. Then remove every blank (`'-'`)
3. Return the list of remaining characters

In [ ]:
def my_ctc_decode(path):
    """CTC decode a path (list of str tokens) -> list of str.
    Step 1: collapse consecutive duplicates.
    Step 2: remove blanks ('-').
    """
    # TODO: implement CTC decoding (about 3 lines)
    raise NotImplementedError('please implement my_ctc_decode')


# ─── Check ───────────────────────────────────────────────
try:
    test_cases = [
        (['H', 'H', '-', 'I', '-'],  ['H', 'I']),
        (['H', '-', 'I', '-', '-'],  ['H', 'I']),
        (['-', 'H', '-', 'I', '-'],  ['H', 'I']),
        (['H', '-', '-', 'I', 'I'],  ['H', 'I']),
        (['H', 'H', 'I', '-', 'I'],  ['H', 'I', 'I']),   # careful: this one decodes to HII, not HI
        (['-', '-', '-'],              []),                # all blanks
        (['A', 'A', 'A'],              ['A']),             # all repeats
    ]
    for path, expected in test_cases:
        result = my_ctc_decode(path)
        assert result == expected, (
            f'path {path} → expected {expected}, got {result}\n'
            f'hint: check the order of collapsing and the blank-removal logic'
        )
    print('✅ all tests passed! my_ctc_decode is correct')
except (NotImplementedError, TypeError):
    print('⏳ implement my_ctc_decode first, then run the check')


## 2. Cross-attention heatmap: which audio frames a decoder token looks at

Every time Whisper's decoder generates a token in an autoregressive (AR) step, it performs cross-attention over the encoder output:

```
score(q, k) = q . k / sqrt(d_k)
alpha = softmax(score)          # shape: (n_tokens, n_frames)
context = alpha @ encoder_out   # weighted aggregation
```

The `alpha` matrix is where the heatmap comes from — rows = output tokens, columns = audio frames, color intensity = attention weight (alpha).
For a well-aligned model the heatmap lights up along the diagonal: sounds spoken earlier map to early frames, later ones to later frames.
Straying off the diagonal means skipped frames, repetition or hallucination.

In [ ]:
# Simulated cross-attention heatmap (synthetic data)
# In real use, replace with actual attention weights captured by a Whisper hook

rng = np.random.default_rng(42)
N_TOKENS = 10   # number of output tokens
N_FRAMES = 40   # number of mel frames (25 ms window, 10 ms hop, about 100 frames/second)

token_labels = list('HELLOWORLD')  # 10 characters as token labels (matching N_TOKENS)
assert len(token_labels) == N_TOKENS
attn = np.zeros((N_TOKENS, N_FRAMES))
for i in range(N_TOKENS):
    center = int(i / N_TOKENS * N_FRAMES)
    spread = 3
    for j in range(N_FRAMES):
        attn[i, j] = np.exp(-0.5 * ((j - center) / spread) ** 2)
    attn[i] += rng.uniform(0, 0.05, N_FRAMES)
    attn[i] /= attn[i].sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: good alignment (diagonal)
im0 = axes[0].imshow(attn, aspect='auto', cmap='viridis', origin='upper')
axes[0].set_xlabel('encoder frames (audio frames)')
axes[0].set_ylabel('output token')
axes[0].set_yticks(range(N_TOKENS))
axes[0].set_yticklabels(list(token_labels))
axes[0].set_title('Cross-attention heatmap\n(diagonal = good alignment)')
plt.colorbar(im0, ax=axes[0], label='attention weight')

# Right: disturbed alignment (simulating hallucination / skipped frames)
attn_bad = attn.copy()
for i in [3, 4, 7]:
    attn_bad[i] = rng.dirichlet(np.ones(N_FRAMES) * 0.3)
im1 = axes[1].imshow(attn_bad, aspect='auto', cmap='viridis', origin='upper')
axes[1].set_xlabel('encoder frames (audio frames)')
axes[1].set_ylabel('output token')
axes[1].set_yticks(range(N_TOKENS))
axes[1].set_yticklabels(list(token_labels))
axes[1].set_title('Cross-attention heatmap\n(some tokens with scattered attention = hallucination risk)')
plt.colorbar(im1, ax=axes[1], label='attention weight')

plt.tight_layout()
plt.show()
print('Left: a clean diagonal. Right: tokens 3/4/7 have diffuse attention, a sign of hallucination or skipped frames')

In [ ]:
# Check the attention matrix's shape and normalization (each row of a softmax output should sum to 1)
assert attn.shape == (N_TOKENS, N_FRAMES), f"attention matrix shape should be ({N_TOKENS}, {N_FRAMES}), got {attn.shape}"
assert np.allclose(attn.sum(axis=1), 1.0, atol=1e-9), "each row of attention weights should sum to 1 (softmax normalization)"
print(f"✅ Cross-attention matrix check passed: shape={attn.shape}, every row sums to 1.0")

## Parameter experiment: turn one knob at a time — change how spread out the attention is and watch the heatmap

Control parameters:
- `spread` (Gaussian width): the larger it is, the more diffuse the attention; `spread=1` is close to a hard alignment, `spread=8` corresponds to an uncertain model
- `noise_level`: background noise in the attention, simulating low-confidence frames

**What to expect**:
- `spread` from 1 → 8: the diagonal widens → the model's sense of which frames belong to the current token gets blurrier and blurrier
- `noise_level` up to 0.15: the background noise rises, the heatmap washes out overall, and the alignment structure fades away
- Compare how the per-row entropy `H = -sum(alpha * log(alpha))` changes for different values of `spread`

In [ ]:
# Parameter experiment: spread & noise_level
rng2 = np.random.default_rng(0)
N_TOKENS2, N_FRAMES2 = 8, 32

def make_attn(spread, noise_level):
    a = np.zeros((N_TOKENS2, N_FRAMES2))
    for i in range(N_TOKENS2):
        center = int(i / N_TOKENS2 * N_FRAMES2)
        for j in range(N_FRAMES2):
            a[i, j] = np.exp(-0.5 * ((j - center) / spread) ** 2)
        a[i] += rng2.uniform(0, noise_level, N_FRAMES2)
        a[i] /= a[i].sum()
    return a

configs = [
    (1,   0.01,  'spread=1, noise=0.01  (hard, clean)'),
    (4,   0.05,  'spread=4, noise=0.05  (medium)'),
    (8,   0.15,  'spread=8, noise=0.15  (scattered, hallucination risk)'),
]

# Cache the results so the plots and the printed entropies use the same data (fixes the bug of drawing from rng2 twice)
results = [make_attn(sp, nl) for sp, nl, _ in configs]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (sp, nl, title), a in zip(axes, configs, results):
    im = ax.imshow(a, aspect='auto', cmap='plasma', origin='upper')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('audio frames')
    ax.set_ylabel('output token')
    entropies = -np.sum(a * np.log(a + 1e-9), axis=1)
    ax.text(0.98, 0.02, f'mean entropy={entropies.mean():.2f}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=8, color='white')
    plt.colorbar(im, ax=ax)

plt.suptitle('Parameter experiment: how attention spread affects the heatmap', y=1.02)
plt.tight_layout()
plt.show()

for (sp, nl, title), a in zip(configs, results):
    H = -np.sum(a * np.log(a + 1e-9), axis=1).mean()
    print(f'spread={sp:2d}, noise={nl:.2f} -> mean attention entropy = {H:.3f}')
print('Higher entropy means the decoder is less sure which frames the current token belongs to: an early warning of hallucination')

## Wrap-up

This lesson enumerated every path with `product(range(V), repeat=T)`, used `ctc_decode` to pick out all the valid CTC paths, and then showed the core alignment idea, "many paths → the same target", as a lattice heatmap.
Visualizing the matrix `alpha = softmax(q·k / sqrt(d_k))` revealed the implicit alignment structure of Whisper's decoder cross-attention; how clean the diagonal is gives you an intuitive early warning of hallucination.
Both pictures map directly onto the inference pipeline `aurora.audio.mel` → encoder → decoder: mel frames on the horizontal axis, the token sequence on the vertical axis.

Next lesson: **L76**, a music theory crash course — the 88 keys of a piano really hold only 12 notes; from pitch and intervals to the chroma wheel and equal temperament, laying the groundwork for music feature engineering.

In [ ]:
# ✏️ Self-review for this lesson
l75_review = {
    "ctc_decode_impl":         None,  # my_ctc_decode() is correct and all 7 tests (including all-blank / all-repeat) pass? True/False
    "ctc_rule_order":          None,  # remember the CTC decoding order: collapse repeats first, then remove blanks — never the other way round? True/False
    "lattice_multipath":       None,  # understand what the lattice plot means: many valid paths decode to the same target sequence? True/False
    "attention_heatmap_read":  None,  # can read a cross-attention heatmap: clean diagonal = good alignment, diffuse = hallucination risk? True/False
    "attention_row_norm":      None,  # understand softmax normalization: each row of the attention matrix sums to 1, high entropy = uncertain? True/False
}

unfilled = [k for k, v in l75_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l75_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L75 all clear! The ASR phase is complete — on to L76: music theory crash course')

---

→ **Next lesson** · [L76 · Music theory crash course](../8_music/L76_music_theory.ipynb)

> Next lesson covers **music theory crash course**: pitch, intervals, the chroma wheel and equal temperament.